# Chapter 5 — RLHF Alignment Pipeline

## Package-based notebook: SFT → Reward Model → DPO → PPO

This revision follows the same package-style pattern as the earlier chapter notebooks:
algorithm code lives under `rlbook.algorithms.alignment`, while notebook/runtime helpers are imported from `rlbook.core`.


In [ ]:
# Colab setup:
# !pip -q install -U transformers datasets accelerate detoxify

In [ ]:
# Setup — run once
# Local:  pip install -e ".[alignment]"
# Colab:
# !pip install -q "git+https://github.com/rl-textbook/reinforcement-learning.git",
# !pip -q install -U transformers datasets accelerate detoxify

import copy
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import display
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from transformers import AutoModelForCausalLM, AutoTokenizer

from rlbook.algorithms.alignment import (
    Config,
    DPOTrainer,
    RewardModel,
    SFTTrainer,
    flush_cuda,
    get_lr_lambda,
    set_seed,
)
from rlbook.core import (
    batch_generate_with_guard,
    build_dataloaders,
    build_preference_probe,
    count_nonfinite,
    dump_text,
    generate_showcase_response,
    held_out_preference_accuracy,
    held_out_raw_preference,
    load_eval_model,
    logs_to_df,
    make_safe_generation_configs,
    make_train_logs,
    plot_bar_metric,
    plot_margin_curve,
    plot_multi_margin,
    plot_multi_scalar,
    plot_scalar_curve,
    plot_winrate_heatmap,
    preference_margin,
    prepare_eval_rows,
    quality_score,
    record_metric,
    refusal_score,
    rm_held_out_accuracy,
    rm_scores,
    run_ppo_stage,
    save_log_csvs,
    select_showcase_index,
    winrate_matrix_from_scores,
)

SEED = 7
set_seed(SEED)
print(f"Seed: {SEED}")



In [ ]:
cfg = Config(seed=SEED)
cfg.pref_train_size=2000
cfg.pref_val_size=200
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = Path("case_study_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"
cfg.pad_token_id = tokenizer.pad_token_id
cfg.max_length = min(cfg.max_length, int(tokenizer.model_max_length))
""" Uncomment out below if you are running locally and/or want to decrease computational load. """
# cfg.pref_train_size = 512
# cfg.pref_val_size = 128

safe_greedy, safe_sample = make_safe_generation_configs(cfg)
train_logs = make_train_logs()

dataset = load_dataset("Anthropic/hh-rlhf", split="train")
data = build_dataloaders(dataset, tokenizer, cfg, device)

sft_rows = data["sft_rows"]
pref_rows = data["pref_rows"]
pref_val_rows = data["pref_val_rows"]
sft_loader = data["sft_loader"]
pref_loader = data["pref_loader"]
pref_val_loader = data["pref_val_loader"]

pref_probe = build_preference_probe(pref_val_loader, device, max_examples=cfg.pref_probe_size)
(output_dir / "config.json").write_text(json.dumps(cfg.to_dict(), indent=2), encoding="utf-8")

print(f"Device: {device} | Model: {cfg.model_name}")
print(f"SFT rows: {len(sft_rows)} | Preference rows: {len(pref_rows)}")


## Stage 1: Supervised Fine-Tuning

In [ ]:
print("=== SFT ===")
base_model = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
base_model.resize_token_embeddings(len(tokenizer))
sft_trainer = SFTTrainer(base_model, lr=cfg.sft_lr)

sft_it = iter(sft_loader)
for step in range(cfg.sft_steps):
    try:
        batch = next(sft_it)
    except StopIteration:
        sft_it = iter(sft_loader)
        batch = next(sft_it)

    loss = sft_trainer.train_step(batch)
    if (step + 1) % 25 == 0 or step == 0 or (step + 1) == cfg.sft_steps:
        record_metric(train_logs, "SFT", step + 1, loss=loss)
    if (step + 1) % 100 == 0 or step == 0:
        print(f"  Step {step + 1}/{cfg.sft_steps} | loss={loss:.4f}")

sft_model = base_model.eval()
sft_pref_margin = preference_margin(sft_model, pref_probe)
record_metric(train_logs, "SFT", cfg.sft_steps, preference_margin=sft_pref_margin)

sft_state = {k: v.cpu() for k, v in sft_model.state_dict().items()}
del sft_trainer
flush_cuda()

print(f"SFT done. Held-out preference margin={sft_pref_margin:.4f}")

## Stage 2: Reward Model

In [ ]:
print("=== RM Training ===")
rm_base = AutoModelForCausalLM.from_pretrained(cfg.model_name)
rm_base.resize_token_embeddings(len(tokenizer))
rm_base.to(device)

hidden_size = rm_base.transformer.wte.weight.size(1)
reward_model = RewardModel(rm_base, hidden_size).to(device)

rm_opt = AdamW(reward_model.parameters(), lr=cfg.rm_lr)
warmup = int(cfg.rm_steps * cfg.warmup_fraction)
rm_sched = LambdaLR(rm_opt, get_lr_lambda(warmup, cfg.rm_steps))

rm_it = iter(pref_loader)
for step in range(cfg.rm_steps):
    try:
        batch = next(rm_it)
    except StopIteration:
        rm_it = iter(pref_loader)
        batch = next(rm_it)

    reward_model.train()
    rc = reward_model(batch["chosen_input_ids"], batch["chosen_attention_mask"])
    rr = reward_model(batch["rejected_input_ids"], batch["rejected_attention_mask"])
    loss = -F.logsigmoid(rc - rr).mean()

    rm_opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
    rm_opt.step()
    rm_sched.step()

    if (step + 1) % 25 == 0 or step == 0 or (step + 1) == cfg.rm_steps:
        acc = (rc > rr).float().mean()
        record_metric(
            train_logs,
            "RM",
            step + 1,
            loss=loss.item(),
            acc=acc.item(),
            lr=rm_sched.get_last_lr()[0],
        )

    if (step + 1) % 20 == 0 or step == 0:
        acc = (rc > rr).float().mean()
        print(f"  Step {step + 1}/{cfg.rm_steps} | loss={loss.item():.4f} | acc={acc:.3f}")

rm_val_acc = rm_held_out_accuracy(reward_model, pref_val_loader)
reward_model.eval()
for p in reward_model.parameters():
    p.requires_grad_(False)

del rm_opt, rm_sched
flush_cuda()

print(f"RM held-out accuracy: {rm_val_acc:.3f}")

## Stage 3: Direct Preference Optimization

In [ ]:
print("=== DPO ===")
dpo_policy = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
dpo_policy.resize_token_embeddings(len(tokenizer))
dpo_policy.load_state_dict(sft_model.state_dict())

ref_policy = copy.deepcopy(dpo_policy).eval()
for p in ref_policy.parameters():
    p.requires_grad_(False)

dpo_trainer = DPOTrainer(dpo_policy, ref_policy, beta=cfg.beta_dpo, lr=cfg.dpo_lr)

dpo_it = iter(pref_loader)
for step in range(cfg.dpo_steps):
    try:
        batch = next(dpo_it)
    except StopIteration:
        dpo_it = iter(pref_loader)
        batch = next(dpo_it)

    loss, train_margin = dpo_trainer.train_step(batch)

    if (step + 1) % cfg.pref_probe_every == 0 or step == 0 or (step + 1) == cfg.dpo_steps:
        heldout_margin = preference_margin(dpo_policy, pref_probe)
        record_metric(
            train_logs,
            "DPO",
            step + 1,
            loss=loss,
            train_margin=train_margin,
            preference_margin=heldout_margin,
        )

    if (step + 1) % 100 == 0:
        dpo_df = logs_to_df(train_logs, "DPO")
        latest_margin = dpo_df["preference_margin"].iloc[-1] if not dpo_df.empty else float("nan")
        print(
            f"  Step {step + 1}/{cfg.dpo_steps} | "
            f"loss={loss:.4f} | heldout_pref_margin={latest_margin:.4f}"
        )

dpo_state = {k: v.cpu() for k, v in dpo_policy.state_dict().items()}
dpo_ref_state = {k: v.cpu() for k, v in ref_policy.state_dict().items()}

del dpo_trainer, ref_policy, dpo_policy
flush_cuda()

print("DPO done.")

## Stage 4: PPO (vanilla)

In [ ]:
ppo_policy = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
ppo_policy.resize_token_embeddings(len(tokenizer))
ppo_policy.load_state_dict(sft_model.state_dict())

ppo_ref = copy.deepcopy(ppo_policy).eval()
for p in ppo_ref.parameters():
    p.requires_grad_(False)

vanilla_trainer = run_ppo_stage(
    "PPO (vanilla)",
    train_logs,
    ppo_policy,
    ppo_ref,
    reward_model,
    tokenizer,
    cfg,
    device,
    sft_rows,
    pref_probe,
    kl_coef=cfg.kl_coef_ppo,
    repetition_penalty=1.0,
    reward_clip=None,
)
vanilla_state = {k: v.cpu() for k, v in vanilla_trainer.policy.state_dict().items()}
vanilla_summary = getattr(vanilla_trainer, "stage_summary", {})

del vanilla_trainer, ppo_policy, ppo_ref
flush_cuda()

print("Vanilla PPO done.")

## Stage 4b: PPO (mitigated)

This matches the current `run.py` defaults: KL coefficient `0.05`, repetition penalty `1.1`, and reward clipping at `±5.0`.

In [ ]:
ppo_m = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
ppo_m.resize_token_embeddings(len(tokenizer))
ppo_m.load_state_dict(sft_model.state_dict())

ppo_m_ref = copy.deepcopy(ppo_m).eval()
for p in ppo_m_ref.parameters():
    p.requires_grad_(False)

mitigated_trainer = run_ppo_stage(
    "PPO (mitigated)",
    train_logs,
    ppo_m,
    ppo_m_ref,
    reward_model,
    tokenizer,
    cfg,
    device,
    sft_rows,
    pref_probe,
    kl_coef=0.1,
    repetition_penalty=1.2,
    reward_clip=2.0,
)
mit_state = {k: v.cpu() for k, v in mitigated_trainer.policy.state_dict().items()}
mitigated_summary = getattr(mitigated_trainer, "stage_summary", {})

del mitigated_trainer, ppo_m, ppo_m_ref
flush_cuda()

print("Mitigated PPO done.")

## Training diagnostics and saved artifacts

In [ ]:
save_log_csvs(train_logs, output_dir)

sft_df = logs_to_df(train_logs, "SFT")
rm_df = logs_to_df(train_logs, "RM")
dpo_df = logs_to_df(train_logs, "DPO")
ppo_v_df = logs_to_df(train_logs, "PPO (vanilla)")
ppo_m_df = logs_to_df(train_logs, "PPO (mitigated)")

plot_scalar_curve(
    sft_df,
    metric="loss",
    title="SFT training loss",
    ylabel="loss",
    ema_beta=0.9,
    save_path=output_dir / "sft_loss_curve.png",
)
plot_scalar_curve(
    rm_df,
    metric="loss",
    title="Reward model training loss",
    ylabel="loss",
    ema_beta=0.9,
    save_path=output_dir / "rm_loss_curve.png",
)
plot_scalar_curve(
    rm_df,
    metric="acc",
    title="Reward model train accuracy",
    ylabel="accuracy",
    ema_beta=0.9,
    save_path=output_dir / "rm_train_accuracy_curve.png",
)
plot_margin_curve(
    dpo_df,
    title="DPO preference margin (held-out probe, EMA-smoothed)",
    baseline=sft_pref_margin,
    save_path=output_dir / "dpo_preference_margin.png",
)
plot_scalar_curve(
    dpo_df,
    metric="train_margin",
    title="DPO implicit reward margin (train batches)",
    ylabel="margin",
    ema_beta=0.9,
    save_path=output_dir / "dpo_train_margin.png",
)
plot_scalar_curve(
    dpo_df,
    metric="loss",
    title="DPO loss",
    ylabel="loss",
    ema_beta=0.9,
    save_path=output_dir / "dpo_loss_curve.png",
)
plot_multi_margin(
    {
        "PPO (vanilla)": ppo_v_df,
        "PPO (mitigated)": ppo_m_df,
    },
    title="PPO preference margins (held-out probe, EMA-smoothed)",
    baseline=sft_pref_margin,
    save_path=output_dir / "ppo_preference_margins.png",
)
plot_multi_scalar(
    {
        "PPO (vanilla)": ppo_v_df,
        "PPO (mitigated)": ppo_m_df,
    },
    metric="approx_kl_mean",
    title="PPO approximate KL per response token",
    ylabel="approx KL",
    ema_beta=0.9,
    save_path=output_dir / "ppo_approx_kl.png",
)
plot_multi_scalar(
    {
        "PPO (vanilla)": ppo_v_df,
        "PPO (mitigated)": ppo_m_df,
    },
    metric="rm_raw_mean",
    title="PPO raw reward-model score",
    ylabel="RM score",
    ema_beta=0.9,
    save_path=output_dir / "ppo_rm_raw_mean.png",
)
plot_multi_scalar(
    {
        "PPO (vanilla)": ppo_v_df,
        "PPO (mitigated)": ppo_m_df,
    },
    metric="reward_mean",
    title="PPO normalized reward mean",
    ylabel="normalized reward",
    ema_beta=0.9,
    save_path=output_dir / "ppo_reward_mean.png",
)
plot_multi_scalar(
    {
        "PPO (vanilla)": ppo_v_df,
        "PPO (mitigated)": ppo_m_df,
    },
    metric="resp_len_mean",
    title="PPO mean response length",
    ylabel="tokens",
    ema_beta=0.9,
    save_path=output_dir / "ppo_response_length.png",
)
plot_multi_scalar(
    {
        "PPO (vanilla)": ppo_v_df,
        "PPO (mitigated)": ppo_m_df,
    },
    metric="loss",
    title="PPO total loss",
    ylabel="loss",
    ema_beta=0.9,
    save_path=output_dir / "ppo_total_loss.png",
)

display(dpo_df.tail())
display(ppo_v_df.tail())
display(ppo_m_df.tail())
print(f"Saved training logs and plots to: {output_dir}")

## Stage 5: Helpfulness and safety evaluation

In [ ]:
models = {
    "SFT": load_eval_model(sft_state, cfg, tokenizer, device),
    "DPO": load_eval_model(dpo_state, cfg, tokenizer, device),
    "PPO (vanilla)": load_eval_model(vanilla_state, cfg, tokenizer, device),
    "PPO (mitigated)": load_eval_model(mit_state, cfg, tokenizer, device),
}

print("Non-finite parameter check:")
nonfinite_counts = {}
for name, model in models.items():
    bad, total = count_nonfinite(model)
    nonfinite_counts[name] = {"bad": int(bad), "total": int(total)}
    print(f"  {name:16s}: {bad}/{total}")

try:
    from detoxify import Detoxify
    det_device = "cuda" if device.type == "cuda" else "cpu"
    det = Detoxify("original", device=det_device)
except Exception as exc:
    det = None
    print(f"Detoxify unavailable: {exc}")

eval_prompts, harm_prompts = prepare_eval_rows(pref_val_rows, pref_rows, cfg, source="pref_val")
print(f"General prompts: {len(eval_prompts)} | Harmful prompts: {len(harm_prompts)}")

In [ ]:
policy_order = ["SFT", "DPO", "PPO (vanilla)", "PPO (mitigated)"]
use_models = {name: models[name] for name in policy_order if name in models}

eval_resps = {}
harm_resps = {}
for name, model in use_models.items():
    eval_resps[name] = batch_generate_with_guard(
        model,
        tokenizer,
        eval_prompts,
        cfg,
        device,
        safe_greedy,
        safe_sample,
        mode="benign",
        batch_size=cfg.eval_batch_size,
    )
    harm_resps[name] = batch_generate_with_guard(
        model,
        tokenizer,
        harm_prompts,
        cfg,
        device,
        safe_greedy,
        safe_sample,
        mode="harmful",
        batch_size=cfg.eval_batch_size,
    )

help_scores = {
    name: rm_scores(eval_prompts, eval_resps[name], tokenizer, reward_model, cfg, device)
    for name in use_models.keys()
}
help_mat = winrate_matrix_from_scores(help_scores)
display(help_mat.round(3))
plot_winrate_heatmap(
    help_mat,
    "Helpfulness win-rate (Reward Model as judge)",
    save_path=output_dir / "helpfulness_winrate_matrix.png",
)
help_mat.to_csv(output_dir / "helpfulness_winrate_matrix.csv")

safety_scores = None
safety_mode = None
if len(harm_prompts) == 0:
    safe_mat = None
    print("No harmful prompts available for safety evaluation.")
elif det is not None:
    tox_scores = {
        name: np.asarray(det.predict(harm_resps[name])["toxicity"], dtype=np.float32)
        for name in use_models.keys()
    }
    safety_scores = {name: (1.0 - tox_scores[name]) for name in use_models.keys()}
    safe_mat = winrate_matrix_from_scores(safety_scores)
    safety_mode = "detoxify"
    display(safe_mat.round(3))
    plot_winrate_heatmap(
        safe_mat,
        "Safety win-rate (1 - Detoxify toxicity)",
        save_path=output_dir / "safety_winrate_matrix.png",
    )
    safe_mat.to_csv(output_dir / "safety_winrate_matrix.csv")
else:
    safety_scores = {
        name: np.asarray([refusal_score(resp) for resp in harm_resps[name]], dtype=np.float32)
        for name in use_models.keys()
    }
    safe_mat = winrate_matrix_from_scores(safety_scores)
    safety_mode = "refusal_proxy"
    display(safe_mat.round(3))
    plot_winrate_heatmap(
        safe_mat,
        "Safety win-rate (refusal regex proxy)",
        save_path=output_dir / "safety_refusal_proxy_winrate_matrix.png",
    )
    safe_mat.to_csv(output_dir / "safety_refusal_proxy_winrate_matrix.csv")

helpfulness_means = {name: float(np.mean(scores)) for name, scores in help_scores.items()}
plot_bar_metric(
    helpfulness_means,
    title="Average helpfulness score (Reward Model)",
    ylabel="mean RM score",
    save_path=output_dir / "mean_helpfulness_rm_score.png",
)

safety_means = None
if safety_scores is not None:
    safety_means = {name: float(np.mean(scores)) for name, scores in safety_scores.items()}
    ylabel = "mean safety score" if safety_mode == "detoxify" else "mean refusal proxy"
    filename = "mean_safety_score.png" if safety_mode == "detoxify" else "mean_refusal_proxy.png"
    plot_bar_metric(
        safety_means,
        title="Average safety comparison",
        ylabel=ylabel,
        save_path=output_dir / filename,
    )

display(pd.DataFrame({"helpfulness_mean": helpfulness_means}))
if safety_means is not None:
    display(pd.DataFrame({"safety_mean": safety_means}))

In [ ]:
ref_model_eval = load_eval_model(dpo_ref_state, cfg, tokenizer, device)
pref_accuracy_summary = {
    "SFT_loglik": float(held_out_raw_preference(models["SFT"], pref_val_loader)),
    "DPO_implicit": float(held_out_preference_accuracy(models["DPO"], ref_model_eval, pref_val_loader, beta=cfg.beta_dpo)),
    "PPO_vanilla_implicit": float(held_out_preference_accuracy(models["PPO (vanilla)"], ref_model_eval, pref_val_loader, beta=cfg.beta_dpo)),
    "PPO_mitigated_implicit": float(held_out_preference_accuracy(models["PPO (mitigated)"], ref_model_eval, pref_val_loader, beta=cfg.beta_dpo)),
}

display(pd.DataFrame({"value": pref_accuracy_summary}).T)
print(f"RM held-out accuracy: {rm_val_acc:.3f}")
print(f"SFT held-out preference margin: {sft_pref_margin:.4f}")

## Qualitative showcase

In [ ]:
aligned_names = [name for name in ["DPO", "PPO (vanilla)", "PPO (mitigated)"] if name in use_models]
showcase_lines = []

if len(aligned_names) >= 2 and len(eval_prompts) > 0:
    benign_idx = select_showcase_index(eval_prompts, eval_resps, help_scores, mode="benign")
    benign_prompt = eval_prompts[benign_idx]
    showcase_lines.append("=== Qualitative showcase (quality-filtered helpfulness spread) ===")
    showcase_lines.append(f"Prompt index: {benign_idx}")
    showcase_lines.append(f"PROMPT:\n{benign_prompt}")
    for name in aligned_names:
        response = generate_showcase_response(
            use_models[name],
            tokenizer,
            reward_model,
            benign_prompt,
            cfg,
            device,
            safe_greedy,
            safe_sample,
            det=det,
            mode="benign",
        )
        score = rm_scores([benign_prompt], [response], tokenizer, reward_model, cfg, device)[0]
        showcase_lines.append("")
        showcase_lines.append("-" * 60)
        showcase_lines.append(f"{name} | RM score={score:.4f} | quality={quality_score(response):.2f}")
        showcase_lines.append(response)

if len(aligned_names) >= 2 and len(harm_prompts) > 0:
    harmful_scores_source = safety_scores if safety_scores is not None else {
        name: np.asarray([refusal_score(resp) for resp in harm_resps[name]], dtype=np.float32)
        for name in use_models.keys()
    }
    harmful_idx = select_showcase_index(harm_prompts, harm_resps, harmful_scores_source, mode="harmful")
    harmful_prompt = harm_prompts[harmful_idx]
    showcase_lines.append("")
    showcase_lines.append("=== Harmful prompt showcase (quality-filtered safety spread) ===")
    showcase_lines.append(f"Prompt index: {harmful_idx}")
    showcase_lines.append(f"PROMPT:\n{harmful_prompt}")
    for name in aligned_names:
        response = generate_showcase_response(
            use_models[name],
            tokenizer,
            reward_model,
            harmful_prompt,
            cfg,
            device,
            safe_greedy,
            safe_sample,
            det=det,
            mode="harmful",
        )
        toxicity = float(det.predict([response])["toxicity"][0]) if det is not None else float("nan")
        showcase_lines.append("")
        showcase_lines.append("-" * 60)
        showcase_lines.append(
            f"{name} | toxicity={toxicity:.4f} | "
            f"refusal={refusal_score(response):.1f} | quality={quality_score(response):.2f}"
        )
        showcase_lines.append(response)

dump_text(output_dir / "qualitative_showcase.txt", "\n".join(showcase_lines))
print("\n".join(showcase_lines[:60]))
print(f"Saved qualitative showcase to: {output_dir / 'qualitative_showcase.txt'}")

## Summary

In [ ]:
summary = {
    "config": cfg.to_dict(),
    "rm_held_out_accuracy": float(rm_val_acc),
    "sft_pref_margin": float(sft_pref_margin),
    "final_pref_margins": {
        "SFT": float(sft_pref_margin),
        "DPO": float(dpo_df["preference_margin"].iloc[-1]) if not dpo_df.empty else None,
        "PPO_vanilla": float(ppo_v_df["preference_margin"].iloc[-1]) if not ppo_v_df.empty else None,
        "PPO_mitigated": float(ppo_m_df["preference_margin"].iloc[-1]) if not ppo_m_df.empty else None,
    },
    "held_out_preference_accuracy": pref_accuracy_summary,
    "helpfulness_means": helpfulness_means,
    "safety_mode": safety_mode,
    "safety_means": safety_means,
    "nonfinite_parameter_counts": nonfinite_counts,
    "ppo_stage_summary": {
        "vanilla": vanilla_summary,
        "mitigated": mitigated_summary,
    },
}
(output_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

display(pd.DataFrame({
    "rm_held_out_accuracy": [summary["rm_held_out_accuracy"]],
    "sft_pref_margin": [summary["sft_pref_margin"]],
    "dpo_pref_margin": [summary["final_pref_margins"]["DPO"]],
    "ppo_v_pref_margin": [summary["final_pref_margins"]["PPO_vanilla"]],
    "ppo_m_pref_margin": [summary["final_pref_margins"]["PPO_mitigated"]],
}))
print(f"Saved summary.json to: {output_dir / 'summary.json'}")

In [ ]:
for name, model in list(models.items()) + [("RM", reward_model), ("REF", ref_model_eval)]:
    del model
flush_cuda()
print("Cleaned up GPU/CPU caches.")